# 📊 ระบบตรวจแบบฝึกหัด: Agentic RAG 4 ชม.
## สำหรับอาจารย์ผู้สอน

---

### วิธีใช้งาน
1. Run cell **Setup** ครั้งแรกครั้งเดียว
2. Run cell **ฟังก์ชันตรวจ** ครั้งเดียว
3. วาง **path ไฟล์ .ipynb** หรือ **GitHub URL** ของนักศึกษา → Run ตรวจ
4. ผลจะ **append** เข้า `4hr_scores.csv` และ `4hr_scores.json` อัตโนมัติ
5. เปลี่ยน path → Run ตรวจคนถัดไป

### เกณฑ์การให้คะแนน (10 คะแนน)

| ขั้นตอน | คะแนน | เกณฑ์ |
|---------|:-----:|-------|
| 1. Chunk + Embed + Search | 3 | Pipeline ทำงาน, Qdrant score ถูกต้อง |
| 2. Agent + Custom Tool | 3 | Tool ทำงาน, Agent เรียกใช้ได้, อธิบาย docstring |
| 3. RAG Agent + Judge | 4 | RAG ตอบครบ 3 ข้อ, LLM-as-Judge, อธิบาย |

## 📦 Setup (Run ครั้งเดียว)

In [ ]:
%%time
import importlib.util, subprocess, sys

def _pip_install(pkg_spec, import_name=None):
    pkg = pkg_spec.split('>=')[0].split('<=')[0].split('==')[0].split('[')[0].strip()
    imp = import_name or {
        'google-genai': 'google.genai', 'google-adk': 'google.adk',
        'sentence-transformers': 'sentence_transformers', 'qdrant-client': 'qdrant_client',
        'langchain-text-splitters': 'langchain_text_splitters',
        'langchain-huggingface': 'langchain_huggingface',
        'scikit-learn': 'sklearn', 'pymupdf': 'fitz',
        'docling-ibm-models': 'docling_ibm_models',
    }.get(pkg, pkg.replace('-', '_'))
    try:
        spec = importlib.util.find_spec(imp)
    except ModuleNotFoundError:
        spec = None
    has_version_constraint = any(op in pkg_spec for op in ('>=', '<=', '==', '>', '<', '!='))
    if spec is not None and not has_version_constraint:
        print(f'  \u23ed\ufe0f  {pkg}: skipped')
        return
    print(f'  \U0001f4e6 {pkg}: installing...', end='', flush=True)
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg_spec],
                       capture_output=True, text=True)
    print(f'\r  \u2705 {pkg}: done' if r.returncode == 0 else f'\r  \u274c {pkg}: failed')
    if r.returncode != 0: print(r.stderr)

for _pkg in ['google-genai', 'nbformat', 'requests', 'langchain-text-splitters']:
    _pip_install(_pkg)

import hashlib, os, json, random, re, csv, requests
from datetime import datetime
import nbformat
from google import genai

# ตั้งค่า Gemini API
try:
    from google.colab import userdata
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
    print('✅ โหลด API Key จาก Colab Secrets')
except Exception:
    GOOGLE_API_KEY = input('🔑 กรุณาวาง Gemini API Key: ')

client = genai.Client(api_key=GOOGLE_API_KEY)
MODEL = 'gemini-3.1-pro-preview'

SCORES_CSV = '4hr_scores.csv'
SCORES_JSON = '4hr_scores.json'

if not os.path.exists(SCORES_CSV):
    with open(SCORES_CSV, 'w', encoding='utf-8', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['Timestamp', 'Student Name', 'Student ID', 'Phone', 'LINE ID',
                         'Step1', 'Step2', 'Step3', 'Total',
                         'AI Suspected', 'Feedback', 'File Path'])
    print(f'📄 สร้างไฟล์ {SCORES_CSV} ใหม่')
else:
    with open(SCORES_CSV, 'r') as f:
        existing = sum(1 for _ in f) - 1
    print(f'📄 {SCORES_CSV} มีข้อมูล {existing} คนแล้ว')

if not os.path.exists(SCORES_JSON):
    with open(SCORES_JSON, 'w') as f:
        json.dump([], f)

print(f'✅ Setup เรียบร้อย | Model: {MODEL}')

## 🔧 ฟังก์ชันตรวจ (Run ครั้งเดียว)

In [ ]:
def read_notebook(filepath_or_url):
    """อ่าน notebook จาก local path หรือ GitHub URL"""
    if filepath_or_url.startswith('http'):
        # GitHub URL → ดึง raw content
        raw_url = filepath_or_url.replace('github.com', 'raw.githubusercontent.com').replace('/blob/', '/')
        print(f'📥 ดึงจาก: {raw_url}')
        resp = requests.get(raw_url)
        resp.raise_for_status()
        nb = nbformat.reads(resp.text, as_version=4)
    else:
        # Local file
        with open(filepath_or_url, 'r', encoding='utf-8') as f:
            nb = nbformat.read(f, as_version=4)
    
    info = {'student_name': '', 'student_id': '', 'phone': '', 'line_id': ''}
    full_content = ''
    
    for i, cell in enumerate(nb.cells):
        if cell.cell_type == 'code':
            for key, var in [('student_id','STUDENT_ID'), ('student_name','STUDENT_NAME'),
                             ('phone','PHONE'), ('line_id','LINE_ID')]:
                m = re.search(rf"{var}\s*=\s*['\"]([^'\"]+)['\"]", cell.source)
                if m:
                    info[key] = m.group(1)
        
        ct = cell.cell_type.upper()
        full_content += f'\n\n=== CELL {i} ({ct}) ===\n{cell.source}'
        if hasattr(cell, 'outputs'):
            for out in cell.outputs:
                if hasattr(out, 'text'):
                    full_content += f'\n--- OUTPUT ---\n{out.text}'
                elif hasattr(out, 'data'):
                    for mime, data in out.data.items():
                        if 'text' in mime:
                            txt = data if isinstance(data, str) else '\n'.join(data)
                            full_content += f'\n--- OUTPUT ---\n{txt}'
    
    return info, full_content




---
## ▶️ ตรวจแบบฝึกหัด (ทีละคน)

เปลี่ยน `NOTEBOOK_PATH` แล้ว **Run cell ด้านล่างซ้ำ** สำหรับแต่ละคน

In [ ]:
# ===== วาง path หรือ GitHub URL =====
NOTEBOOK_PATH = ''  # local path หรือ GitHub URL
# ตัวอย่าง local: '/content/drive/MyDrive/submissions/student_hw.ipynb'
# ตัวอย่าง GitHub: 'https://github.com/user/repo/blob/main/homework.ipynb'

assert NOTEBOOK_PATH, '❌ กรุณาวาง path หรือ GitHub URL!'

# 1. อ่าน notebook
info, content = read_notebook(NOTEBOOK_PATH)
print(f'👤 {info["student_name"]} ({info["student_id"]})')
print(f'📱 {info["phone"]} | 💬 {info["line_id"]}')

# 🔍 ตรวจซ้ำ
already_graded = False
if os.path.exists(SCORES_JSON):
    with open(SCORES_JSON, 'r') as f:
        prev = json.load(f)
    for p in prev:
        if p.get('student_id') == info['student_id']:
            already_graded = True
            print(f'⚠️  รหัส {info["student_id"]} เคยตรวจแล้ว (คะแนน: {p.get("total_score", "?")})')
            break

if already_graded:
    overwrite = input('🔄 ต้องการตรวจใหม่? (y/n): ').strip().lower()
    if overwrite != 'y':
        print('⏭️  ข้าม')
        raise SystemExit()
    prev = [p for p in prev if p.get('student_id') != info['student_id']]
    with open(SCORES_JSON, 'w', encoding='utf-8') as f:
        json.dump(prev, f, ensure_ascii=False, indent=2)
    rows = []
    with open(SCORES_CSV, 'r', encoding='utf-8') as f:
        rows = list(csv.reader(f))
    with open(SCORES_CSV, 'w', encoding='utf-8', newline='') as f:
        writer = csv.writer(f)
        for row in rows:
            if len(row) > 2 and row[2] == info['student_id']:
                continue
            writer.writerow(row)
    print(f'🗑️  ลบคะแนนเดิมแล้ว')

# 2. สร้างเฉลย
expected = generate_expected(info['student_id'])
print(f'🔑 เฉลย: chunks={expected["num_chunks"]}, query="{expected["query"][:30]}..."')

# 3. ตรวจด้วย Gemini
print(f'🤖 กำลังตรวจด้วย {MODEL}...')
grade = grade_with_gemini(info['student_id'], content, expected)

# 4. แสดงผล
print(f'\n{"="*60}')
print(f'📊 ผลการตรวจ: {info["student_name"]} ({info["student_id"]})')
print(f'{"="*60}')
print(f'  ขั้นตอน 1 (Pipeline):  {grade["step1_score"]}/3  — {grade["step1_feedback"]}')
print(f'  ขั้นตอน 2 (Agent):    {grade["step2_score"]}/3  — {grade["step2_feedback"]}')
print(f'  ขั้นตอน 3 (RAG):      {grade["step3_score"]}/4  — {grade["step3_feedback"]}')
print(f'  {"─"*56}')
print(f'  🏆 คะแนนรวม: {grade["total_score"]}/10')
print(f'  💬 {grade["overall_feedback"]}')
if grade.get('is_ai_suspected'):
    print(f'  ⚠️  สงสัยใช้ AI: {grade["ai_reason"]}')

# 5. Append
append_result(info, grade, NOTEBOOK_PATH)

## 📊 ดูคะแนนทั้งหมด

In [ ]:
import csv

with open(SCORES_CSV, 'r', encoding='utf-8') as f:
    reader = csv.reader(f)
    header = next(reader)
    rows = list(reader)

print(f'📊 ตรวจไปแล้ว {len(rows)} คน\n')
print(f'{"#":>3} {"ชื่อ":<20} {"รหัส":<12} {"S1":>4} {"S2":>4} {"S3":>4} {"รวม":>5} {"AI?":>4}')
print('=' * 65)

for idx, row in enumerate(rows, 1):
    name, sid = row[1], row[2]
    s1, s2, s3, total = row[5], row[6], row[7], row[8]
    ai = '⚠️' if row[9] == 'True' else '✅'
    print(f'{idx:>3} {name:<20} {sid:<12} {s1:>4} {s2:>4} {s3:>4} {total:>5} {ai:>4}')

if rows:
    scores = [float(r[8]) for r in rows]
    print(f'\n📈 สถิติ: ต่ำสุด={min(scores):.1f}, สูงสุด={max(scores):.1f}, เฉลี่ย={sum(scores)/len(scores):.1f}')

## 💾 ดาวน์โหลดไฟล์คะแนน

In [ ]:
try:
    from google.colab import files
    files.download(SCORES_CSV)
    print(f'✅ ดาวน์โหลด {SCORES_CSV}')
except:
    print(f'📄 ไฟล์คะแนนอยู่ที่: {os.path.abspath(SCORES_CSV)}')